In [ ]:
#test_vis_params tests different parameters for saving landsat images to png
#CONCLUSION: for Landsat 4,5,7 use {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 1.5, "gamma":1.4}
#            for Lansdat 8,9 use {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 1.5, "gamma":1.4}
#       See examples C:\Users\andyb\Documents\U\GEE-Courses\data\landsat_vis_exports
#
#Grok prompt:
#geemap python landsat iterate over a selection of vis_params and save results as png
#See also: test_cloud_cover_local, test_minmax

# --------------------------------------------------------------
# 1. Install / import
# --------------------------------------------------------------
# !pip install -q geemap  # run once in Colab / Jupyter

import ee
import geemap
import os
from pathlib import Path

# --------------------------------------------------------------
# 2. Authenticate & initialize Earth Engine
# --------------------------------------------------------------
ee.Authenticate()   # only needed the first time
ee.Initialize()

In [ ]:
print(geemap.__version__)

# do not run - switched it to markdown --------------------------------------------------------------
# 3. Define the area of interest (AOI)
# --------------------------------------------------------------
# Example: a polygon around Anchorage, AK (you can replace with any geometry)
aoi = ee.Geometry.Polygon(
    [[[-149.5, 61.0],
      [-149.5, 61.5],
      [-148.5, 61.5],
      [-148.5, 61.0],
      [-149.5, 61.0]]])

# --------------------------------------------------------------
# 4. Build the Landsat image collection
# --------------------------------------------------------------
landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA') #L2')   # Landsat 8
           .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_TOA')) #L2'))  # Landsat 9
           .filterBounds(aoi)
           .filterDate('2025-06-01', '2025-09-01')   # summer season
           #.sort('CLOUD_COVER')
           .first())                         # pick the least cloudy image

# Apply scaling factors (SR data are in 1/10000)
def apply_scale_factors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True) \
                .addBands(thermalBands, None, True)

#landsat = apply_scale_factors(landsat)
landsat

In [ ]:
aoi=ee.Geometry.Polygon( #Margerie - wider than terminus aoi to show snow in 20140903 image
    [[[-137.21, 58.99],
      [-137.04, 58.99],
      [-137.04, 59.06],
      [-137.21, 59.06],
      [-137.21, 58.99]]])

#landsat = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20210508')  # Example SF
#landsat = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140903') #Most clear Margerie, but scene has a lot of cloud
#file_prefix='MargerieL08_20140903'
#landsat = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140207') #Margerie Feb
#file_prefix='MargerieL08_20140207'

#landsat = ee.Image('LANDSAT/LT05/C02/T1_TOA/LT05_060019_19921015') #Margerie 
#file_prefix='MargerieL05_19921015'

#landsat = ee.Image('LANDSAT/LE07/C02/T1_TOA/LE07_060019_20010813') #Margerie
#file_prefix='MargerieL07_20010813'

landsat = ee.Image('LANDSAT/LC09/C02/T1_TOA/LC09_059019_20240805') #Margerie
file_prefix='MargerieL09_20240805'

#aoi = ee.Geometry.Rectangle([-137.2,58.81,-137.0,58.86])  #JHI
#landsat = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140903') #scene has a lot of cloud
#file_prefix='JohnsHopkinsL08_20140903'
#landsat = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140207') #scene has a lot of cloud
#file_prefix='JohnsHopkinsL08_20140207'


In [ ]:
# --------------------------------------------------------------
# 5. List of visualization parameters to iterate over
# --------------------------------------------------------------
#GD_Landsat_02_Download currently using:
#Add the Landsat TOA image to the map (visualize with true color bands) TOP OF ATMOSPHERE
#vis_params = {'bands': ['B4', 'B3', 'B2'],'min': 0.05,'max': 1.6}
#for Landsat 4 and 5, GEEDiT uses bands:['B3','B2','B1'],gamma:1.5,min:0,max:0.8
#vis_params45 = {'bands': ['B3', 'B2', 'B1'],'min': 0.05,'max': 1.6}

#for surface reflectance
vis_params_list = [
    {"name": "true_color",
     "params": {"bands": ['SR_B4', 'SR_B3', 'SR_B2'], "min": 0, "max": 0.3}},
    {"name": "false_color_nir",
        "params": {"bands": ['SR_B5', 'SR_B4', 'SR_B3'], "min": 0, "max": 0.3}},
    {"name": "agriculture",
        "params": {"bands": ['SR_B6', 'SR_B5', 'SR_B2'], "min": 0, "max": 0.3}},
    {"name": "thermal",
        "params": {"bands": ['ST_B10'], "min": 290, "max": 320, "palette": ['blue', 'white', 'red']}}
]

#for top of atmosphere Landsat 8 or 9
vis_params_list = [
    {"name": "true_00-03","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 0.3}}, #true_color_toa #standard for non-snow images
    {"name": "true_00-07","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 0.7}},
    {"name": "true_00-08","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 0.8}},
    {"name": "true_00-09","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 0.9}}, #AKB FAVORITE for Margerie and JH Sept
    {"name": "true_00-10","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 1.0}},
    {"name": "true_00-15","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 1.5}}, #AKB best Feb
    {"name": "true_01-12","params": {"bands": ['B4', 'B3', 'B2'], "min": 0.1, "max": 1.2}},
    {"name": "true_02-12","params": {"bands": ['B4', 'B3', 'B2'], "min": 0.2, "max": 1.2}}, #based off JHI histogram from test_landsat_histogram, see Histogram JHI Unscaled TOA.png
    {"name": "true_-05-15","params": {"bands": ['B4', 'B3', 'B2'], "min": -0.5, "max": 1.5}}, #starting with min of -.5 = shift everything brighter
    {"name": "true_-03-13","params": {"bands": ['B4', 'B3', 'B2'], "min": -0.3, "max": 1.3}},
    {"name": "true_-03-11","params": {"bands": ['B4', 'B3', 'B2'], "min": -0.3, "max": 1.1}},
    {"name": "true_-01-08","params": {"bands": ['B4', 'B3', 'B2'], "min": -0.1, "max": 0.8}},
    {"name": "true_-01-09","params": {"bands": ['B4', 'B3', 'B2'], "min": -0.1, "max": .9}},
    {"name": "true_-01-11","params": {"bands": ['B4', 'B3', 'B2'], "min": -0.1, "max": 1.1}},
    {"name": "false_00-04","params": {"bands": ['B5', 'B4', 'B3'], "min": 0, "max": 0.4}}, #false_color_nir_toa
    {"name": "false_00-10","params": {"bands": ['B5', 'B4', 'B3'], "min": 0, "max": 1}},
    {"name": "swir_00-04","params": {"bands": ['B6', 'B5', 'B4'], "min": 0, "max": 0.4}}, #swir_nir_red_toa
    {"name": "swir_00-10","params": {"bands": ['B6', 'B5', 'B4'], "min": 0, "max": 1}},
    {"name": "thermal_290-320","params": {"bands": ['B10'],"min": 290, "max": 320,"palette": ['blue', 'cyan', 'yellow', 'red'] }},#thermal_toa
    {"name": "thermal_200-290","params": {"bands": ['B10'],"min": 200, "max": 290,"palette": ['blue', 'cyan', 'yellow', 'red'] }},
    {"name": "thermal_230-290","params": {"bands": ['B10'],"min": 230, "max": 290,"palette": ['blue', 'cyan', 'yellow', 'red'] }}, #AKB FAVORITE thermal Feb
    {"name": "thermal_250-290","params": {"bands": ['B10'],"min": 250, "max": 290,"palette": ['blue', 'cyan', 'yellow', 'red'] }}, #AKB FAVORITE thermal Sept
    {"name": "thermal_270-300","params": {"bands": ['B10'],"min": 270, "max": 300,"palette": ['blue', 'cyan', 'yellow', 'red'] }}
]

#for top of atmosphere Landsat 4 or 5 or 7
vis_params_list = [
    {"name": "true_00-03","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 0.3}}, #true_color_toa #standard for non-snow images
    {"name": "true_00-07","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 0.7}},
    {"name": "true_00-08","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 0.8}},
    {"name": "true_00-09","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 0.9}}, #AKB FAVORITE for Margerie and JH Sept
    {"name": "true_00-10","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 1.0}},
    {"name": "true_00-15","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 1.5}}, #AKB best Feb
    {"name": "true_01-12","params": {"bands": ['B3', 'B2', 'B1'], "min": 0.1, "max": 1.2}},
    {"name": "true_02-12","params": {"bands": ['B3', 'B2', 'B1'], "min": 0.2, "max": 1.2}}, #based off JHI histogram from test_landsat_histogram, see Histogram JHI Unscaled TOA.png
    {"name": "true_-05-15","params": {"bands": ['B3', 'B2', 'B1'], "min": -0.5, "max": 1.5}}, #starting with min of -.5 = shift everything brighter
    {"name": "true_-03-13","params": {"bands": ['B3', 'B2', 'B1'], "min": -0.3, "max": 1.3}},
    {"name": "true_-03-11","params": {"bands": ['B3', 'B2', 'B1'], "min": -0.3, "max": 1.1}},
    {"name": "true_-01-08","params": {"bands": ['B3', 'B2', 'B1'], "min": -0.1, "max": 0.8}},
    {"name": "true_-01-09","params": {"bands": ['B3', 'B2', 'B1'], "min": -0.1, "max": .9}},
    {"name": "true_-01-11","params": {"bands": ['B3', 'B2', 'B1'], "min": -0.1, "max": 1.1}},    
    {"name": "false_00-04","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 0.4}}, #false_color_nir_toa
    {"name": "false_00-10","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 1}},
    {"name": "swir_00-04","params": {"bands": ['B5', 'B4', 'B3'], "min": 0, "max": 0.4}}, #swir_nir_red_toa
    {"name": "swir_00-10","params": {"bands": ['B5', 'B4', 'B3'], "min": 0, "max": 1}},
    #wider band, may have different min max
    {"name": "thermal_290-320","params": {"bands": ['B6'],"min": 290, "max": 320,"palette": ['blue', 'cyan', 'yellow', 'red'] }},#thermal_toa
    {"name": "thermal_200-290","params": {"bands": ['B6'],"min": 200, "max": 290,"palette": ['blue', 'cyan', 'yellow', 'red'] }},
    {"name": "thermal_230-290","params": {"bands": ['B6'],"min": 230, "max": 290,"palette": ['blue', 'cyan', 'yellow', 'red'] }}, #AKB FAVORITE thermal Feb
    {"name": "thermal_250-290","params": {"bands": ['B6'],"min": 250, "max": 290,"palette": ['blue', 'cyan', 'yellow', 'red'] }}, #AKB FAVORITE thermal Sept
    {"name": "thermal_270-300","params": {"bands": ['B6'],"min": 270, "max": 300,"palette": ['blue', 'cyan', 'yellow', 'red'] }}
]

#Experiment with gamma:
#gamma>1 (e.g., 1.2–1.6): Brightens the image. This is often used to bring out details in shadows without overexposing the already bright areas.
#gamma=1 No gamma correction is applied (linear mapping).
#gamma<1 (e.g., 0.6–0.9): Darkens the image (increases contrast in dark areas).
#gamma Landsat 4,5,7
vis_params_list = [ #starting from AKB best Feb
    {"name": "true_00-15gamma08","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 1.5, "gamma":0.8}},
    {"name": "true_00-15gamma10","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 1.5, "gamma":1.0}}, #same as no gamma
    {"name": "true_00-15gamma12","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 1.5, "gamma":1.2}},
    {"name": "true_00-15gamma14","params": {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 1.5, "gamma":1.4}}, #AKB best of all
]

#gamma landsat 8 or 9
#for top of atmosphere Landsat 8 or 9
vis_params_list = [
    {"name": "true_00-15gamma08","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 1.5, "gamma":0.8}},
    {"name": "true_00-15gamma10","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 1.5, "gamma":1.0}},
    {"name": "true_00-15gamma12","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 1.5, "gamma":1.2}},
    {"name": "true_00-15gamma14","params": {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 1.5, "gamma":1.4}}, #AKB best of all
]
#OBSOLETE NOTE: Ideal for display would be something like a mosaic with range 0-.3 for darker areas and 0-1.5 for brighter 
#e.g. half histogram goes from 0-.15 and half .15-1.5. Not explaining this well... see image_visualization.ipynb
#ACTUALLY...what I'm trying to describe was gamma - set to 1.4.

# --------------------------------------------------------------
# 6. Output folder
# --------------------------------------------------------------
out_dir = Path(r'C:\Users\andyb\Documents\U\GEE-Courses\data\landsat_vis_exports')#"landsat_vis_exports")
out_dir.mkdir(exist_ok=True)

In [ ]:
#6a test map
item=vis_params_list[0]
name   = item["name"]
vis    = item["params"]
# Create a temporary Map object (so each export is independent)
#Map = geemap.Map(center=[61.25, -149.0], zoom=9)
#Map = geemap.Map(center=[59.04, -137.07], zoom=13)
Map=geemap.Map()
Map.centerObject(aoi, 13)
# Add the Landsat layer with the current vis params
Map.addLayer(aoi,{},'aoi')
Map.addLayer(landsat.clip(aoi), vis, name)
# OPTIONAL: add a basemap for context
#Map.add_basemap('ROADMAP')
Map

In [ ]:
# --------------------------------------------------------------
# 7. Iterate, render, and export PNGs
# --------------------------------------------------------------
for item in vis_params_list:
    name   = item["name"]
    vis    = item["params"]
    
    # Create a temporary Map object (so each export is independent)
    #Map = geemap.Map(center=[61.25, -149.0], zoom=9)
    Map = geemap.Map(center=[59.04, -137.07], zoom=13)

    # Add the Landsat layer with the current vis params
    Map.addLayer(landsat.clip(aoi), vis, name)
    
    # OPTIONAL: add a basemap for context
    #Map.add_basemap('ROADMAP')
    
    # Define the export file path
    png_path = out_dir / f"{file_prefix}_{name}.png"
    
    # Export the map view as PNG
    #   region = aoi (clip to the AOI)
    #   dimensions = 1200x1200 (adjust as you like)
    #   scale = 30 m (Landsat native resolution)
#    Map.to_png(
#        filename=str(png_path),
#        region=aoi,
#        dimensions=(1200, 1200),
#        scale=30,
#        crs='EPSG:4326'
#    )
#    geemap.get_image_thumbnail(image,filename,vis_params=vis_params,dimensions=2000,crs='EPSG:32608') #UTM Zone 8N (EPSG:32608) 3338 Alaska Albers as a test
    geemap.get_image_thumbnail(landsat.clip(aoi),str(png_path),vis_params=vis,dimensions=1200,crs='EPSG:32608') #UTM Zone 8N (EPSG:32608) 3338 Alaska Albers as a test

    print(f"Exported: {png_path}")

print("\nAll PNGs saved to:", out_dir.resolve())